# Skill 31: Scaled Dot-Product Attention

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Compute self-attention by hand and in code, and justify every term of the equation.

## Objectives
- Understand query, key and value tensors and their compatible dimensions.
- Scale scores by $\sqrt{d_k}$ to control magnitude before softmax.
- Compute attention weights and use them to form a weighted sum of values.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$
Dividing by $\sqrt{d_k}$ keeps the score variance at 1 (raw variance is $d_k$), so softmax does not saturate.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import math
import torch

def scaled_dot_product_attention(q, k, v, mask=None):
    # q: (B, Tq, D), k: (B, Tk, D), v: (B, Tk, Dv)
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
    if mask is not None:
        # Boolean True means "blocked" in this implementation.
        scores = scores.masked_fill(mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ v
    return output, weights

torch.manual_seed(7)
q = torch.randn(2, 4, 8)
k = torch.randn(2, 4, 8)
v = torch.randn(2, 4, 8)
out, weights = scaled_dot_product_attention(q, k, v)
print("output:", out.shape)       # (2, 4, 8)
print("weights:", weights.shape)  # (2, 4, 4)
print("row sums:", weights.sum(-1))


## What to check
**Expected result:** Output shape `(2, 4, 8)`; weights `(2, 4, 4)`; every weight row sums to 1.

## Pitfalls
Boolean `True` means *blocked* in this code (PyTorch's `F.scaled_dot_product_attention` uses the opposite). A fully masked row gives NaN. Softmax must be over the last axis.